In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from multiprocessing import Process, Queue, Pool
import emcee
from astropy.cosmology import FlatLambdaCDM
from astropy import units as u
import warnings
import sys
warnings.filterwarnings("ignore")

import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
plt.rc('font', size=24)

In [2]:
#Helper function for our escape library
def cosmology(cosmology):
    case = cosmology.name
    if case == 'Flatw0waCDM':
        return [cosmology.Om0, cosmology.w0, cosmology.wa, cosmology.h]
    
    elif case == 'FlatwCDM':
        return [cosmology.Om0, cosmology.w0, cosmology.h]

    elif case == 'wCDM':
        return [cosmology.Om0, cosmology.Ode0, cosmology.w0,cosmology.h]
        
    elif case == 'LambdaCDM':
        return [cosmology.Om0, cosmology.Ode0, cosmology.h]

    elif case == 'FlatLambdaCDM':
        return [cosmology.Om0, cosmology.h]


In [3]:
#please rename these for your file paths
path_to_function_libraries = "/your_path_to/Escape_Library/Function_Libraries/"
path_to_galaxy_data = "/your_path_to/Escape_Library/Data/"
path_to_Zv_calibration='/your_path_to/Escape_Library/AGAMA_Zv_calibration'

if path_to_function_libraries not in sys.path:
    sys.path.insert(0, path_to_function_libraries)
    
from Dispersion_funcs import make_sigma_to_r200_carlberg,calculate_converged_sigma_data

Omega_m = 0.3
Omega_L = 1-Omega_m
h0 = 0.7

cosmo_name = 'FlatLambdaCDM'
cosmo = FlatLambdaCDM(H0=h0*100.0,Om0=Omega_m,name = cosmo_name)
cosmo_params = cosmology(cosmo)

galaxy_positional_data = np.genfromtxt(os.path.join(path_to_galaxy_data, 'Rines_galaxy_data.txt'))

In this example we assume the cluster we observe is A7. We use Rines 2013 and 2016 HeCS and HeCS-SZ data, and a cluster position specified in RA, DEC, and z. The assumed error on velocities in 30 km/s, update as needed.

In [4]:
cluster_position = (2.9385416666666666, 32.41569444444444, 0.106)
sigma_to_r200 = make_sigma_to_r200_carlberg(cosmo)

result = calculate_converged_sigma_data(
        cluster_positional_data=cluster_position,
        galaxy_positional_data=galaxy_positional_data,
        cosmo_params=cosmo_params,
        cosmo_name=cosmo_name,
        sigma_to_r200=sigma_to_r200,
        velocity_errors=30.0,
    )

print('Dispersion Median:', result['sigma_hat'])
print('+1 sigma:', result['ci84'])
print('-1 sigma:', result['ci16'])

Dispersion Median: 907.1041768373728
+1 sigma: 946.7062116921677
-1 sigma: 867.3505450753612
